# Download counts

- All resources:https://assets.nemoarchive.org/dat-ztfn3cc

- readME with all resources. https://data.nemoarchive.org/other/grant/broad_mccarroll_HD_2024/mccarroll/transcriptome/sncell/10x_v3.1/human/processed/counts/README_10x_counts_updated_2025-07-03.txt

- counts preprocessed: https://assets.nemoarchive.org/collection/nemo:dat-oj75u6x 
- cell meta
- donor meta: 

In [ ]:
%%bash
BASE="https://data.nemoarchive.org/other/grant/broad_mccarroll_HD_2024/mccarroll/transcriptome/sncell"
SAVE_FOLDER="/home/gdallagl/myworkdir/XDP/data/HD_Handsaker/adata/NEMO"
LOG="$SAVE_FOLDER/wget.log"
mkdir -p "$SAVE_FOLDER"

RETRY_OPT=""
wget --help 2>&1 | grep -q "retry-on-http-error" && \
  RETRY_OPT="--retry-on-http-error=429,500,502,503,504"

# robust single-file downloader --< let's trust Caude ??= maybe chekc in future...
dl () {
  local url="$1" tries=0 max=8
  until wget -c -nv -a "$LOG" -P "$SAVE_FOLDER" $RETRY_OPT --tries=3 --waitretry=20 "$url"; do
    tries=$((tries+1))
    if [ "$tries" -ge "$max" ]; then
      echo "FAILED: $url" | tee -a "$LOG"; return 1
    fi
    echo "  retry $tries for $(basename "$url") in $((30*tries))s ..."
    sleep $((30*tries))
  done
  echo "OK: $(basename "$url")"
}

FAILED=0
for f in donor_metadata.txt \
         10x_village_cell_metadata.txt \
         10x_singledonor_cell_metadata.txt \
         10x_village_cell_metadata_with_SPN_subtypes.txt \
         10x_singledonor_cell_metadata_with_SPN_subtypes.txt; do
  dl "$BASE/10x_v3.1/human/processed/other/$f" || FAILED=1
done

for f in pacbio_deepdives_cells_unfiltered.txt pacbio_metadata_deepdives_unfiltered.txt; do
  dl "$BASE/pacbio/human/processed/other/$f" || FAILED=1
done

# counts matrices (recursive): -c makes it skip every finished .h5 on rerun
wget -r -l1 -np -nd -c -e robots=off -A "*umi_*.h5" \
     $RETRY_OPT --tries=5 --waitretry=30 \
     -nv -a "$LOG" -P "$SAVE_FOLDER" \
     "$BASE/10x_v3.1/human/processed/counts/" || FAILED=1

echo; echo "=== $(ls "$SAVE_FOLDER"/*.h5 2>/dev/null | wc -l) h5 files, $(ls "$SAVE_FOLDER"/*.txt 2>/dev/null | wc -l) txt files ==="
[ "$FAILED" -eq 0 ] && echo "ALL DONE" || echo "some downloads FAILED — rerun this cell later, or check $LOG"





# doownlaod geneome used for mpping
# OTHERWISE PROBLMES WITH GNE ENAME COVERSION
curl -L ftp://ftp.ensembl.org/pub/release-89/gtf/homo_sapiens/Homo_sapiens.GRCh38.89.gtf.gz -o "$SAVE_FOLDER/Homo_sapiens.GRCh38.89.gtf.gz"


OK: donor_metadata.txt
OK: 10x_village_cell_metadata.txt
OK: 10x_singledonor_cell_metadata.txt
OK: 10x_village_cell_metadata_with_SPN_subtypes.txt
OK: 10x_singledonor_cell_metadata_with_SPN_subtypes.txt
OK: pacbio_deepdives_cells_unfiltered.txt
OK: pacbio_metadata_deepdives_unfiltered.txt

=== 89 h5 files, 7 txt files ===
ALL DONE


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 39.7M  100 39.7M    0     0  10.8M      0  0:00:03  0:00:03 --:--:-- 10.8M


# Buids h5ad

In [ ]:
import glob, os
import pandas as pd
import scanpy as sc
import anndata as ad
import h5py
import numpy as np
import scipy.sparse as sp

FOLDER = "/home/gdallagl/myworkdir/XDP/data/HD_Handsaker/adata/NEMO"


In [ ]:
# ---------- 1) load and concatenate all count matrices ----------

import re

# name files strnage, use regex to accot for that

def read_mccarroll_h5(path):
    with h5py.File(path, "r") as f:
        g = f["matrix"]
        X = sp.csc_matrix(
            (g["data"][:], g["indices"][:], g["indptr"][:]),
            shape=tuple(g["shape"][:]),
        ).T.tocsr()
        barcodes = [b.decode() if isinstance(b, bytes) else b for b in g["barcodes"][:]]
        names = [n.decode() if isinstance(n, bytes) else n for n in g["features"]["name"][:]]
    obs = pd.DataFrame(index=barcodes)
    var = pd.DataFrame(index=names)
    return ad.AnnData(X=X, obs=obs, var=var)

def reaction_prefix(path):
    """h5 filename stem -> metadata PREFIX (matches CELL_BARCODE format).
    CHDI_Village1_rxn1                  -> Village1_rxn1
    S06758_HD_Caudate_DeepDive_rxn5     -> S06758_Caudate_DeepDive_rxn5
    S04002_10X_rxn2_merged              -> S04002_rxn2
    """
    stem = os.path.basename(path).split(".umi_")[0]
    if stem.startswith("CHDI_"):
        stem = stem[len("CHDI_"):]
    stem = stem.replace("_HD_", "_").replace("_10X_", "_")
    stem = re.sub(r"_merged$", "", stem)
    return stem

adatas = []
for path in sorted(glob.glob(f"{FOLDER}/*umi_*.h5"))[:]:
    a = read_mccarroll_h5(path)
    a.var_names_make_unique()

    rxn = reaction_prefix(path)
    a.obs["reaction"] = rxn
    a.obs["experiment"] = "village" if "Village" in rxn else "deepdive"
    a.obs_names = rxn + "_" + a.obs_names.astype(str)   # matches metadata CELL_BARCODE
    adatas.append(a)

adata = ad.concat(adatas, join="outer", merge="same")
adata.obs["CELL_BARCODE"] = adata.obs_names
del adatas
print(adata)
display(adata.obs)
display(adata.var)


AnnData object with n_obs × n_vars = 1088454 × 39411
    obs: 'reaction', 'experiment', 'CELL_BARCODE'


,reaction,experiment,CELL_BARCODE
Village14_rxn1_ATCACTTCAGAGGCAT,Village14_rxn1,village,Village14_rxn1_ATCACTTCAGAGGCAT
Village14_rxn1_GGCTGTGCAGGAATCG,Village14_rxn1,village,Village14_rxn1_GGCTGTGCAGGAATCG
Village14_rxn1_CTCACTGAGGGCTTCC,Village14_rxn1,village,Village14_rxn1_CTCACTGAGGGCTTCC
Village14_rxn1_AATCGACCATCTCCCA,Village14_rxn1,village,Village14_rxn1_AATCGACCATCTCCCA
Village14_rxn1_TCGACCTTCTGCTTTA,Village14_rxn1,village,Village14_rxn1_TCGACCTTCTGCTTTA
...,...,...,...
S12365_Caudate_DeepDive_rxn4_GTGATGTAGTATTAGG,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_GTGATGTAGTATTAGG
S12365_Caudate_DeepDive_rxn4_AGGATAAAGAATCCCT,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_AGGATAAAGAATCCCT
S12365_Caudate_DeepDive_rxn4_AGCGTATGTCGTGATT,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_AGCGTATGTCGTGATT
S12365_Caudate_DeepDive_rxn4_CAATACGCAGAGTGTG,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_CAATACGCAGAGTGTG


""
A1BG
A1BG-AS1
A1CF
A2M
A2M-AS1
...
hsa-mir-550a-2
snoR1
snoU83B
snoZ196


In [4]:
# ---------- 2) per-cell metadata (donor, cell type, QC) ----------
cellmeta = pd.concat([
    pd.read_csv(f"{FOLDER}/10x_village_cell_metadata_with_SPN_subtypes.txt", sep="\t"),
    pd.read_csv(f"{FOLDER}/10x_singledonor_cell_metadata_with_SPN_subtypes.txt", sep="\t"),
])

print(cellmeta.columns.tolist())
display(cellmeta.head(2))
display(cellmeta.tail(2))
print(adata.obs_names[:2].tolist())

cellmeta = cellmeta[['CELL_BARCODE', 'DONOR']].copy()

print("barcode overlap:", adata.obs_names.isin(cellmeta.index).mean())  # should be near 1.0
#print("missing barcodes:", adata.obs_names[~adata.obs_names.isin(cellmeta.index)])

adata.obs = adata.obs.merge(cellmeta, on="CELL_BARCODE", how="left")

print("cells wiht no donor: ")
display(adata.obs[adata.obs["DONOR"].isna()])
adata = adata[~adata.obs["DONOR"].isna()].copy() #ATTENTION, REMOVE CELL WIHT NO DONOR

print(adata.obs.isna().sum())
adata.obs

/tmp/ipykernel_166716/2771665810.py:3: DtypeWarning: Columns (5,7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(f"{FOLDER}/10x_village_cell_metadata_with_SPN_subtypes.txt", sep="\t"),


['PREFIX', 'CBC', 'CELL_BARCODE', 'predClass', 'max.prob', 'predClass_SPN_DI', 'max.prob_SPN_DI', 'predClass_SPN_EN', 'max.prob_SPN_EN', 'predClass_SPN_MP', 'max.prob_SPN_MP', 'DONOR', 'NUM_GENIC_READS', 'NUM_TRANSCRIPTS', 'NUM_GENES']


,PREFIX,CBC,CELL_BARCODE,predClass,max.prob,predClass_SPN_DI,max.prob_SPN_DI,predClass_SPN_EN,max.prob_SPN_EN,predClass_SPN_MP,max.prob_SPN_MP,DONOR,NUM_GENIC_READS,NUM_TRANSCRIPTS,NUM_GENES
0,Village1_rxn1,TACGTCCCACTAACCA,Village1_rxn1_TACGTCCCACTAACCA,SPN,1.0,direct,1.000,noneccentric,1.0,patch,1.0,S17094,196791,53217,7831
1,Village1_rxn1,CCCTCAATCACCCTCA,Village1_rxn1_CCCTCAATCACCCTCA,SPN,1.0,direct,0.999,noneccentric,1.0,matrix,1.0,S02574,144034,38141,7514


,PREFIX,CBC,CELL_BARCODE,predClass,max.prob,predClass_SPN_DI,max.prob_SPN_DI,predClass_SPN_EN,max.prob_SPN_EN,predClass_SPN_MP,max.prob_SPN_MP,DONOR,NUM_GENIC_READS,NUM_TRANSCRIPTS,NUM_GENES
365245,S06758_Caudate_DeepDive_rxn5,CTAGACAAGCTCAGAG,S06758_Caudate_DeepDive_rxn5_CTAGACAAGCTCAGAG,oligodendrocyte,0.999,NaN,NaN,NaN,NaN,NaN,NaN,S06758,7163,2347,1294
365246,S06758_Caudate_DeepDive_rxn5,AGTCATGCAGCGATTT,S06758_Caudate_DeepDive_rxn5_AGTCATGCAGCGATTT,microglia,1.000,NaN,NaN,NaN,NaN,NaN,NaN,S06758,5764,1798,1185


['Village14_rxn1_ATCACTTCAGAGGCAT', 'Village14_rxn1_GGCTGTGCAGGAATCG']
barcode overlap: 0.0


/home/gdallagl/.pyenv/versions/3.11.8/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


cells wiht no donor: 


,reaction,experiment,CELL_BARCODE,DONOR
0,Village14_rxn1,village,Village14_rxn1_ATCACTTCAGAGGCAT,NaN
15,Village14_rxn1,village,Village14_rxn1_TCGAACACAGTCCCGA,NaN
17,Village14_rxn1,village,Village14_rxn1_CGTCCATCACCGTGGT,NaN
24,Village14_rxn1,village,Village14_rxn1_CATAAGCGTTCCGCTT,NaN
26,Village14_rxn1,village,Village14_rxn1_TAGGTACGTTTACTGG,NaN
...,...,...,...,...
1088344,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_ATCAGGTTCGCTACGG,NaN
1088349,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_GAACTGTAGGTGCATG,NaN
1088355,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_TATTCCAAGGTCACCC,NaN
1088357,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_CAGGTATAGGTGCTGA,NaN


reaction        0
experiment      0
CELL_BARCODE    0
DONOR           0
dtype: int64


,reaction,experiment,CELL_BARCODE,DONOR
1,Village14_rxn1,village,Village14_rxn1_GGCTGTGCAGGAATCG,S15245
2,Village14_rxn1,village,Village14_rxn1_CTCACTGAGGGCTTCC,S06720
3,Village14_rxn1,village,Village14_rxn1_AATCGACCATCTCCCA,S06758
4,Village14_rxn1,village,Village14_rxn1_TCGACCTTCTGCTTTA,S13957
5,Village14_rxn1,village,Village14_rxn1_CAGATCATCTCATGGA,S00002
...,...,...,...,...
1088449,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_GTGATGTAGTATTAGG,S12365
1088450,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_AGGATAAAGAATCCCT,S12365
1088451,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_AGCGTATGTCGTGATT,S12365
1088452,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_CAATACGCAGAGTGTG,S12365


In [5]:
# ---------- 3) donor metadata (age, sex, grade) ----------
donormeta = pd.read_csv(f"{FOLDER}/donor_metadata.txt", sep="\t")
display(donormeta.head())

donormeta = donormeta.drop(columns=["Donor"])

# ATTNETION: control have Nan for many column !!!

adata.obs = adata.obs.merge(donormeta, left_on ="DONOR" , right_on="SID", how="left")
display(adata.obs.head())


,Donor,SID,Age,Sex,Status,CAG,PMI,VS_Grade,Onset/Motor,Onset/Cog,CAP,CAP-100
0,1,S05202,67,Male,Case,15/42,23.36,HD-3,40.0,42.0,559,123.9
1,2,S12365,81,Female,Case,17/40,17.33,HD-0,75.0,NaN,514,124.8
2,3,S07681,69,Male,Case,16/42,18.16,HD-1,NaN,NaN,575,127.6
3,4,S06758,37,Female,Case,18/43,16.40,HD-1,NaN,NaN,346,74.1
4,5,S04002,58,Male,Case,17/42,13.45,HD-2,30.0,NaN,484,107.2


/home/gdallagl/.pyenv/versions/3.11.8/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


,reaction,experiment,CELL_BARCODE,DONOR,SID,Age,Sex,Status,CAG,PMI,VS_Grade,Onset/Motor,Onset/Cog,CAP,CAP-100
0,Village14_rxn1,village,Village14_rxn1_GGCTGTGCAGGAATCG,S15245,S15245,61,Male,Control,NaN,22.75,NaN,NaN,NaN,NaN,NaN
1,Village14_rxn1,village,Village14_rxn1_CTCACTGAGGGCTTCC,S06720,S06720,59,Female,Control,NaN,23.78,NaN,NaN,NaN,NaN,NaN
2,Village14_rxn1,village,Village14_rxn1_AATCGACCATCTCCCA,S06758,S06758,37,Female,Case,18/43,16.40,HD-1,NaN,NaN,346,74.1
3,Village14_rxn1,village,Village14_rxn1_TCGACCTTCTGCTTTA,S13957,S13957,58,Female,Control,NaN,25.70,NaN,NaN,NaN,NaN,NaN
4,Village14_rxn1,village,Village14_rxn1_CAGATCATCTCATGGA,S00002,S00002,57,Female,Control,17/32,NaN,NaN,NaN,NaN,NaN,NaN


### PacBio CAG lenght info:  ONLY diseased + ONLY deep dive

CELL_BARCODE    A unique cell barcode for this cell (can be matched to the 10x transcriptome data, contains the PREFIX).
UMI             The unique molecular identifier for the transcript used for this cell (these files are deduplicated only one transcript per cell is listed).
NREADS          The number of pacbio reads for this transcript used in the analysis.
READLENGTH      The median length of the pacbio reads.
CAGLENGTH       The consensus CAG repeat length for this cell/transcript.
CATEGORY        A code assigned based on properties of the observed transcript, sometimes used in downstream analysis.
                The possible categories are:
                  POLYA      The reads end at the A-rich sequence at the exon1/intron1 boundary (presumably a processed transcript).
                  EXON2      The reads appear to be spliced from HTT exon 1 to exon 2 (and possibly beyond).
                  EXON3      The reads appear to be spliced from HTT exon 1 to exon 3 (skipping exon 2).
                  INTRON1    The reads extend into intron 1 (unprocessed transcript).
                  OTHER      The reads do not appear to match any of the categories above.
ALIGNSTART      Consensus alignment start (from reads that were aligned to the reference genome).
ALIGNEND        Consensus alignment end (from reads that were aligned to the reference genome).
DONOR           The ID of the donor for this cell.
REGION          The brain region for this cell.
CELLTYPE        The cell type assigned to this cell.

In [6]:
# ---------- 4) per-cell CAG repeat length (deep dives only) ----------
cag = pd.read_csv(f"{FOLDER}/pacbio_deepdives_cells_unfiltered.txt", sep="\t")
display(cag.head())

# ATTNETION: control have Nan for many column !!!

cag = cag[["CELL_BARCODE", "CAGLENGTH", "CATEGORY"]].copy() #.drop(columns=["UMI", "ALIGNSTART", "ALIGNEND", "DONOR", "CELLTYPE"]).copy()

adata.obs = adata.obs.merge(cag, on="CELL_BARCODE", how="left")
adata.obs


,CELL_BARCODE,UMI,NREADS,READLENGTH,CAGLENGTH,CATEGORY,ALIGNSTART,ALIGNEND,DONOR,REGION,CELLTYPE
0,S05202_Caudate_DeepDive_rxn1_AATGGCTTCGTTGTTT,AAACAACGAAGCCATT.GTATAATCAGGC,918,769,43,EXON2,3074853.0,3075088.0,S05202,CD,astrocyte
1,S05202_Caudate_DeepDive_rxn1_GTCTACCCAGGTGTTT,AAACACCTGGGTAGAC.ACATATTTGAAG,1130,690,15,EXON2,3074853.0,3075088.0,S05202,CD,SPN
2,S05202_Caudate_DeepDive_rxn1_GACTATGGTCGAGTTT,AAACTCGACCATAGTC.CCTGACATTGGT,221,517,42,POLYA,3074853.0,3075088.0,S05202,CD,polydendrocyte
3,S05202_Caudate_DeepDive_rxn1_TGTGCGGAGGGTCTTT,AAAGACCCTCCGCACA.AACCAAAAAATA,177,1106,42,EXON2,3074853.0,3075088.0,S05202,CD,interneuron
4,S05202_Caudate_DeepDive_rxn1_AGATGAATCAGTCTTT,AAAGACTGATTCATCT.ATGGTTAATCCT,142,438,15,POLYA,3074853.0,3075088.0,S05202,CD,microglia


/home/gdallagl/.pyenv/versions/3.11.8/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


,reaction,experiment,CELL_BARCODE,DONOR,SID,Age,Sex,Status,CAG,PMI,VS_Grade,Onset/Motor,Onset/Cog,CAP,CAP-100,CAGLENGTH,CATEGORY
0,Village14_rxn1,village,Village14_rxn1_GGCTGTGCAGGAATCG,S15245,S15245,61,Male,Control,NaN,22.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Village14_rxn1,village,Village14_rxn1_CTCACTGAGGGCTTCC,S06720,S06720,59,Female,Control,NaN,23.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Village14_rxn1,village,Village14_rxn1_AATCGACCATCTCCCA,S06758,S06758,37,Female,Case,18/43,16.40,HD-1,NaN,NaN,346,74.1,NaN,NaN
3,Village14_rxn1,village,Village14_rxn1_TCGACCTTCTGCTTTA,S13957,S13957,58,Female,Control,NaN,25.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Village14_rxn1,village,Village14_rxn1_CAGATCATCTCATGGA,S00002,S00002,57,Female,Control,17/32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
911059,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_GTGATGTAGTATTAGG,S12365,S12365,81,Female,Case,17/40,17.33,HD-0,75.0,NaN,514,124.8,NaN,NaN
911060,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_AGGATAAAGAATCCCT,S12365,S12365,81,Female,Case,17/40,17.33,HD-0,75.0,NaN,514,124.8,NaN,NaN
911061,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_AGCGTATGTCGTGATT,S12365,S12365,81,Female,Case,17/40,17.33,HD-0,75.0,NaN,514,124.8,NaN,NaN
911062,S12365_Caudate_DeepDive_rxn4,deepdive,S12365_Caudate_DeepDive_rxn4_CAATACGCAGAGTGTG,S12365,S12365,81,Female,Case,17/40,17.33,HD-0,75.0,NaN,514,124.8,NaN,NaN


### Manage gene names

In [7]:
# remove genes wiht too few cells
print(adata)
sc.pp.filter_genes(adata, min_cells=10)
print(adata)


AnnData object with n_obs × n_vars = 911064 × 39411
    obs: 'reaction', 'experiment', 'CELL_BARCODE', 'DONOR', 'SID', 'Age', 'Sex', 'Status', 'CAG', 'PMI', 'VS_Grade', 'Onset/Motor', 'Onset/Cog', 'CAP', 'CAP-100', 'CAGLENGTH', 'CATEGORY'
AnnData object with n_obs × n_vars = 911064 × 36943
    obs: 'reaction', 'experiment', 'CELL_BARCODE', 'DONOR', 'SID', 'Age', 'Sex', 'Status', 'CAG', 'PMI', 'VS_Grade', 'Onset/Motor', 'Onset/Cog', 'CAP', 'CAP-100', 'CAGLENGTH', 'CATEGORY'
    var: 'n_cells'


In [8]:
import gzip, re

GTF_PATH = f"{FOLDER}/Homo_sapiens.GRCh38.89.gtf.gz"

rows = []
with gzip.open(GTF_PATH, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if fields[2] != "gene":
            continue
        attrs = fields[8]
        gid = re.search(r'gene_id "([^"]+)"', attrs)
        gname = re.search(r'gene_name "([^"]+)"', attrs)
        gid = gid.group(1) if gid else None
        rows.append({"Ensembl_ID": gid, "gene_symbol": gname.group(1) if gname else gid})

gene_df = pd.DataFrame(rows)
gene_df["gene_symbol"] = gene_df["gene_symbol"].fillna(gene_df["Ensembl_ID"])

# same ambiguity as before: some symbols map to >1 Ensembl_ID (e.g. PAR-linked genes) -- keep first, deterministic
n_dup_symbols = gene_df["gene_symbol"].duplicated().sum()
print(f"{n_dup_symbols} duplicate gene_symbol entries in the release-89 GTF (keeping first)")

symbol_to_ensembl = (
    gene_df.drop_duplicates(subset="gene_symbol", keep="first")
           .set_index("gene_symbol")["Ensembl_ID"]
)

adata.var["gene_symbol"] = adata.var_names          # preserve the symbol before var_names gets overwritten
adata.var["Ensembl_ID"] = adata.var_names.map(symbol_to_ensembl)

n_lost = adata.var["Ensembl_ID"].isna().sum()
print(f"{n_lost} / {adata.n_vars} genes have no Ensembl_ID match")

adata = adata[:, adata.var["Ensembl_ID"].notna()].copy()

adata.var_names = adata.var["Ensembl_ID"].values
adata.var_names.name = None
adata.var_names_make_unique()

adata.var


1662 duplicate gene_symbol entries in the release-89 GTF (keeping first)
0 / 36943 genes have no Ensembl_ID match


,n_cells,gene_symbol,Ensembl_ID
ENSG00000121410,17772,A1BG,ENSG00000121410
ENSG00000268895,64935,A1BG-AS1,ENSG00000268895
ENSG00000148584,11036,A1CF,ENSG00000148584
ENSG00000175899,139303,A2M,ENSG00000175899
ENSG00000245105,28299,A2M-AS1,ENSG00000245105
...,...,...,...
ENSG00000283172,11,hsa-mir-548d-1,ENSG00000283172
ENSG00000263690,205,hsa-mir-548d-2,ENSG00000263690
ENSG00000281010,3975,snoR1,ENSG00000281010
ENSG00000252945,24,snoU83B,ENSG00000252945


In [9]:
adata.layers["counts"] = adata.X.copy()


In [ ]:
adata.obs["village"] = np.where(adata.obs["experiment"] == "village", adata.obs["reaction"].str.split("_").str[0], np.nan)

In [ ]:
adata.obs.index = adata.obs['CELL_BARCODE'].values


In [ ]:
# Calculate qc
    # attention: need raw counts in .X
adata.var["mt"] = adata.var.gene_symbol.str.startswith("MT-") # mitochondrial genes, ATTNETION: the names are in "gene_symbol"
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

# Save

In [ ]:
# ---------- 5) save ----------
for col in adata.obs.columns:
    if not pd.api.types.is_numeric_dtype(adata.obs[col]):
        adata.obs[col] = adata.obs[col].astype(object).fillna("NA").astype(str)

adata.write_h5ad(f"{os.path.dirname(FOLDER)}/HD_Handsaker_merged.h5ad")
adata


/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


AnnData object with n_obs × n_vars = 911064 × 36943
    obs: 'reaction', 'experiment', 'CELL_BARCODE', 'DONOR', 'SID', 'Age', 'Sex', 'Status', 'CAG', 'PMI', 'VS_Grade', 'Onset/Motor', 'Onset/Cog', 'CAP', 'CAP-100', 'CAGLENGTH', 'CATEGORY'
    var: 'n_cells', 'gene_symbol', 'Ensembl_ID'
    layers: 'counts'